# 1014 29일차

## 1. 전이학습

전이학습(Transfer Learning)은 다른 데이터로 학습된 모델을 가져와 내 데이터에 맞게 학습하는 방법 (`keras73_vgg16_13_cifar10` ~ `keras73_vgg16_18_cat_dog`)

큰 모델을 처음부터 학습시키려면 데이터와 시간이 많이 듦. 이미 학습된 가중치에서 시작함

1. 사전 학습 모델(Pre-Trained Model) : ImageNet 같은 큰 데이터로 미리 학습된 모델. `VGG16` 등
2. Feature Extraction : 사전 학습 모델을 고정된 특징 추출기로 씀
3. Fine Tuning : 사전 학습 모델의 가중치에서 시작해 내 데이터로 더 학습함

- 두 방식 모두 `include_top=False`로 분류 부분을 빼고, 클래스 수에 맞는 `Dense`를 붙임

### 1-1. Feature Extraction

사전 학습 모델을 동결하고 뒤에 붙인 `Dense`만 학습하는 방식

1. `vgg16.trainable = False` : Conv 블록 가중치를 고정함. 입력에서 특징만 뽑음
2. 붙인 `Dense` : 뽑은 특징으로 분류를 학습함. 이 층들만 가중치가 바뀜

- 학습할 가중치가 적어 epoch당 시간이 짧음

### 1-2. Fine Tuning

사전 학습 모델의 가중치까지 같이 학습해 내 데이터에 맞게 조금씩 바꾸는 방식

1. `vgg16.trainable = True` : 사전 학습 모델의 가중치도 학습함
2. 작은 `learning_rate` : 학습된 가중치를 조금씩만 바꿈

- 학습할 가중치가 많아 epoch당 시간이 길어짐

#### cf) Fine Tuning에서 learning_rate가 크면

뒤에 붙인 `Dense`는 랜덤 가중치에서 시작해서 학습 초반 기울기가 큼. `learning_rate`가 크면 사전 학습 모델의 가중치도 크게 바뀌어 학습이 안 될 수 있음

- loss가 ln(클래스 수), acc가 1 / 클래스 수에서 멈춤. 모델이 모든 클래스에 같은 확률을 냄

## 2. 층별 동결

층마다 `trainable`을 정해 일부 층만 학습에서 빼는 방법 (`keras74_1_layers` ~ `keras74_4_vgg16_partial_freezing`)

`model.trainable = False`는 모델 전체를 동결함. 층 하나씩 정하면 원하는 층만 고정할 수 있음

1. `model.layers` : 층 리스트. 번호로 층 하나를 꺼냄
2. `layer.trainable` : 층마다 동결 여부. `False`로 바꾸면 그 층만 동결
3. `layer.name`·`type(layer).__name__` : 층 이름과 종류

- `Sequential`에 사전 학습 모델을 넣으면 `model.layers`에서 층 하나(`Functional`)로 보임
- 사전 학습 모델 안쪽 층은 그 모델의 `layers`로 꺼냄. `include_top=False`인 VGG16은 `InputLayer`, `Conv2D` 13개, `MaxPooling2D` 5개
- 가중치 개수는 가중치가 있는 층 수 × 2(`kernel`, `bias`). Pooling·`Flatten`은 가중치가 없음
- 동결한 층의 파라미터는 `summary()`의 Non-trainable params로 들어감

## 3. keras.applications

Keras가 제공하는 사전 학습 모델 모음 (`keras75_All_TransferLearning`, `keras76_all_T_01_cifar10`)

같은 데이터에 사전 학습 모델만 바꿔 끼워 성능을 비교할 수 있음

1. VGG : VGG16, VGG19
2. ResNet : ResNet50·101·152와 각 V2
3. DenseNet : DenseNet121·169·201
4. Inception : InceptionV3, InceptionResNetV2, Xception
5. MobileNet : MobileNet, MobileNetV2, MobileNetV3Small·Large
6. NASNet : NASNetMobile, NASNetLarge
7. EfficientNet : EfficientNetB0·B1·B2

- 모델마다 기본 입력 크기가 다름. Xception은 `(299, 299, 3)`
- 모델 객체 리스트를 `for`로 돌며 같은 층을 붙이면 여러 모델을 한 번에 비교할 수 있음. 모델 이름은 `model.name`

#### cf) weights와 trainable_weights 개수가 다른 모델

`trainable = True`여도 `weights`가 `trainable_weights`보다 많은 모델이 있음

- `BatchNormalization`의 `moving_mean`·`moving_variance`는 역전파로 바뀌지 않아서 `trainable_weights`에서 빠짐
- VGG에는 `BatchNormalization`이 없어서 두 개수가 같음